# SentinelNet v4.0 — T4 GPU Training
**Author:** @who_is_the_black_hat

**Architecture:** Embedding → CNN (k=2,3,5) → Transformer Encoder → Multi-Head Classifier

**Outputs:** `label` · `threat_type` · `action_hint` · `confidence` · `reasoning`

**Steps:**
1. Runtime > Change runtime type > **T4 GPU**
2. Cell 1 — GPU check
3. Upload `threat_v4_labeled.jsonl` to Google Drive root (run `scripts/label_and_prepare_v4.py` first)
4. Cell 2 — Load data from Drive
5. Cell 3 — Model definition
6. Cell 4 — Train
7. Cell 5 — Save + Download
8. Local pe replace:
   ```bash
   cp ~/Downloads/sentinel_threat_net.pt /home/kali/osints/models/ml_engine/
   cp ~/Downloads/sentinel_vocab.json    /home/kali/osints/models/ml_engine/
   ```

In [ ]:
# Cell 1 — GPU Check
import torch
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA :', torch.cuda.is_available())
print('PyTorch:', torch.__version__)

In [ ]:
# Cell 2 — Load Data (Kaggle)
import json, os
from pathlib import Path
from collections import Counter, defaultdict

# Kaggle pe dataset /kaggle/input/ mein hota hai
PATHS = [
    '/kaggle/input/sentinel-data/threat_v4_final.jsonl',
    '/kaggle/input/threat-v4-final/threat_v4_final.jsonl',
    '/kaggle/working/threat_v4_final.jsonl',
]
DATA_FILE = next((p for p in PATHS if Path(p).exists()), None)
if not DATA_FILE:
    import glob
    found = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
    DATA_FILE = found[0] if found else None
if not DATA_FILE:
    raise FileNotFoundError('threat_v4_final.jsonl nahi mila! Dataset add karo.')

print(f'Using: {DATA_FILE}')

THREAT_LABELS = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
samples = []
with open(DATA_FILE) as f:
    for line in f:
        try:
            s = json.loads(line.strip())
            if s.get('text') and s.get('label') in THREAT_LABELS:
                samples.append(s)
        except: pass

print(f'Loaded : {len(samples):,} samples')
print(f'Labels : {dict(Counter(s["label"] for s in samples))}')
has_type   = sum(1 for s in samples if s.get('threat_type'))
has_action = sum(1 for s in samples if s.get('action_hint'))
print(f'threat_type  coverage: {has_type}/{len(samples)}')
print(f'action_hint  coverage: {has_action}/{len(samples)}')


In [ ]:
# Cell 3 — SentinelNet v4.0 Definition
import re, math, time
from collections import Counter, defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

THREAT_LABELS = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
LABEL2IDX     = {l: i for i, l in enumerate(THREAT_LABELS)}
IDX2LABEL     = {i: l for i, l in enumerate(THREAT_LABELS)}

THREAT_TYPES  = ['recon', 'web_vuln', 'breach', 'malware', 'phishing',
                 'apt', 'insider', 'misconfig', 'social_eng', 'unknown']
TYPE2IDX      = {t: i for i, t in enumerate(THREAT_TYPES)}
IDX2TYPE      = {i: t for i, t in enumerate(THREAT_TYPES)}

ACTION_HINTS  = ['monitor', 'patch_now', 'block_ip', 'escalate', 'investigate',
                 'notify_team', 'collect_evidence', 'no_action']
HINT2IDX      = {h: i for i, h in enumerate(ACTION_HINTS)}
IDX2HINT      = {i: h for i, h in enumerate(ACTION_HINTS)}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class SentinelTokenizer:
    SECURITY_TERMS = {
        'ransomware','malware','exploit','backdoor','trojan','botnet',
        'phishing','spearphishing','zero-day','zeroday','apt','c2',
        'command-and-control','lateral-movement','privilege-escalation',
        'sqli','xss','csrf','ssrf','rce','lfi','xxe','ssti','cors','jwt',
        'oauth','infostealer','keylogger','rootkit','fileless','mimikatz',
        'cobalt-strike','metasploit','cve','cvss','mitre','osint','exfiltration',
        'persistence','darkweb','tor','onion',
    }
    PAD_IDX = 0; UNK_IDX = 1

    def __init__(self, max_vocab=12000):
        self.max_vocab  = max_vocab
        self.word2idx   = {'<PAD>': 0, '<UNK>': 1}
        self.vocab_size = 2

    def _tokenize(self, text):
        return [w for w in re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', text.lower()) if len(w) >= 2]

    def build_vocab(self, texts):
        counter = Counter()
        for t in texts: counter.update(self._tokenize(t))
        for term in self.SECURITY_TERMS: counter[term] = counter.get(term, 0) + 1000
        for word, _ in counter.most_common(self.max_vocab - 2):
            if word not in self.word2idx: self.word2idx[word] = len(self.word2idx)
        self.vocab_size = len(self.word2idx)
        print(f'Vocab: {self.vocab_size} tokens')

    def encode(self, text, max_len=300):
        ids = [self.word2idx.get(t, self.UNK_IDX) for t in self._tokenize(text)[:max_len]]
        return ids + [self.PAD_IDX] * (max_len - len(ids))

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({'word2idx': self.word2idx, 'max_vocab': self.max_vocab}, f)


class ThreatDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=300):
        self.encodings    = [tokenizer.encode(s['text'], max_len) for s in samples]
        self.labels       = [LABEL2IDX[s['label']] for s in samples]
        self.threat_types = [TYPE2IDX.get(s.get('threat_type', 'unknown'), 9) for s in samples]
        self.action_hints = [HINT2IDX.get(s.get('action_hint', 'monitor'), 0) for s in samples]

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encodings[idx],    dtype=torch.long),
            torch.tensor(self.labels[idx],        dtype=torch.long),
            torch.tensor(self.threat_types[idx],  dtype=torch.long),
            torch.tensor(self.action_hints[idx],  dtype=torch.long),
        )


class SentinelNet(nn.Module):
    VERSION = '4.0'; AUTHOR = 'who_is_the_black_hat'

    def __init__(self, vocab_size, embed_dim=128, num_filters=128,
                 kernels=(3,5,7), nhead=4, tf_layers=2, ff_dim=512,
                 dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_drop  = nn.Dropout(dropout)
        self.convs     = nn.ModuleList([
            nn.Sequential(nn.Conv1d(embed_dim, num_filters, k, padding=k//2), nn.GELU())
            for k in kernels
        ])
        cnn_out = num_filters * len(kernels)   # 384
        self.cnn_proj  = nn.Linear(cnn_out, embed_dim)
        self.layer_norm  = nn.LayerNorm(embed_dim)
        self.head_label  = self._head(embed_dim, len(THREAT_LABELS), dropout)
        self.head_type   = self._head(embed_dim, len(THREAT_TYPES),  dropout)
        self.head_action = self._head(embed_dim, len(ACTION_HINTS),  dropout)
        for n, p in self.named_parameters():
            if 'weight' in n and p.dim() >= 2: nn.init.xavier_uniform_(p)
            elif 'bias' in n: nn.init.zeros_(p)

    @staticmethod
    def _head(in_d, out_d, dr):
        return nn.Sequential(nn.Dropout(dr), nn.Linear(in_d, in_d//2), nn.GELU(), nn.Linear(in_d//2, out_d))

    def _pos_enc(self, x):
        B, L, D = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, D, 2, device=x.device).float() * (-math.log(10000.0)/D))
        pe  = torch.zeros(L, D, device=x.device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:D//2])
        return x + pe.unsqueeze(0)

    def forward(self, x):
        emb = self.pos_drop(self.embedding(x)).transpose(1, 2)        # (B,D,L)
        # CNN parallel convolutions + global max pool
        pooled = torch.cat([c(emb).max(dim=2).values for c in self.convs], dim=-1)  # (B,3*F)
        ctx = torch.relu(self.cnn_proj(pooled))                       # (B,D)
        return self.head_label(ctx), self.head_type(ctx), self.head_action(ctx)

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {'params': p, 'size_mb': round(p*4/1024/1024, 2)}


print(f'SentinelNet v4.0 defined | Device: {DEVICE}')
print(f'Outputs: label({len(THREAT_LABELS)}) + threat_type({len(THREAT_TYPES)}) + action_hint({len(ACTION_HINTS)})')

In [ ]:
# Cell 4 — Train
# Stratified 85/15 split
cls_idx = defaultdict(list)
for i, s in enumerate(samples): cls_idx[LABEL2IDX[s['label']]].append(i)
train_idx, val_idx = [], []
for idxs in cls_idx.values():
    n = max(1, int(len(idxs)*0.15))
    val_idx.extend(idxs[:n]); train_idx.extend(idxs[n:])

tr_s = [samples[i] for i in train_idx]
vl_s = [samples[i] for i in val_idx]

MAX_LEN = 300; BATCH = 256; EPOCHS = 30; PATIENCE = 5

tokenizer = SentinelTokenizer(max_vocab=12000)
tokenizer.build_vocab([s['text'] for s in tr_s])

PIN = DEVICE.type == 'cuda'
NW  = 2 if DEVICE.type == 'cuda' else 0
tr_dl = DataLoader(ThreatDataset(tr_s, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=True,  num_workers=NW, pin_memory=PIN)
vl_dl = DataLoader(ThreatDataset(vl_s, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=False, num_workers=NW, pin_memory=PIN)

model = SentinelNet(tokenizer.vocab_size).to(DEVICE)
print(f'Params: {model.info()["params"]:,} | Size: {model.info()["size_mb"]} MB')

# Class weights (label head only)
tr_labels = [LABEL2IDX[s['label']] for s in tr_s]
dist = Counter(tr_labels)
w = torch.tensor([len(tr_labels)/(4*dist.get(i,1)) for i in range(4)], dtype=torch.float).to(DEVICE)

crit_label  = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
crit_type   = nn.CrossEntropyLoss(label_smoothing=0.05)
crit_action = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=8e-4, steps_per_epoch=len(tr_dl), epochs=EPOCHS)

def f1_weighted(preds, labels):
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for p, l in zip(preds, labels):
        if p == l: tp[l] += 1
        else: fp[p] += 1; fn[l] += 1
    f1s, ws = [], []
    for c in range(4):
        pr = tp[c]/(tp[c]+fp[c]+1e-8); rc = tp[c]/(tp[c]+fn[c]+1e-8)
        f1s.append(2*pr*rc/(pr+rc+1e-8)); ws.append(labels.count(c))
    return sum(f*w for f,w in zip(f1s,ws))/sum(ws)

best_f1, best_state, no_imp = 0.0, None, 0
print(f'Training on {DEVICE}...')

for epoch in range(1, EPOCHS+1):
    model.train(); tl = 0
    for xb, yb_l, yb_t, yb_a in tr_dl:
        xb, yb_l, yb_t, yb_a = xb.to(DEVICE), yb_l.to(DEVICE), yb_t.to(DEVICE), yb_a.to(DEVICE)
        optimizer.zero_grad()
        l_l, l_t, l_a = model(xb)
        loss = 0.6*crit_label(l_l, yb_l) + 0.2*crit_type(l_t, yb_t) + 0.2*crit_action(l_a, yb_a)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); tl += loss.item()
    tl /= len(tr_dl)

    model.eval(); vl = 0; preds = []; lbls = []
    with torch.no_grad():
        for xb, yb_l, yb_t, yb_a in vl_dl:
            xb, yb_l, yb_t, yb_a = xb.to(DEVICE), yb_l.to(DEVICE), yb_t.to(DEVICE), yb_a.to(DEVICE)
            l_l, l_t, l_a = model(xb)
            vl += (0.6*crit_label(l_l,yb_l) + 0.2*crit_type(l_t,yb_t) + 0.2*crit_action(l_a,yb_a)).item()
            preds.extend(l_l.argmax(1).cpu().tolist()); lbls.extend(yb_l.cpu().tolist())
    vl /= len(vl_dl)
    acc = sum(p==l for p,l in zip(preds,lbls))/len(lbls)
    f1  = f1_weighted(preds, lbls)
    print(f'Epoch {epoch:2d} | train={tl:.4f} | val={vl:.4f} | acc={acc:.2%} | f1={f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_imp = 0
    else:
        no_imp += 1
        if no_imp >= PATIENCE: print(f'Early stop @ epoch {epoch}'); break

model.load_state_dict(best_state)
print(f'\nBest F1: {best_f1:.4f}')

In [ ]:
# Cell 5 — Save + Download
# Kaggle — files /kaggle/working/ mein save hote hain, download button se lo

checkpoint = {
    'model_state':  model.state_dict(),
    'model_config': model.info(),
    'hyperparams':  {
        'embed_dim':   128,
        'num_filters': 128,
        'tf_layers':   2,
        'dropout':     0.3,
        'max_len':     MAX_LEN,
    },
    'author':        'who_is_the_black_hat',
    'version':       '4.0',
    'github':        'https://github.com/Mrsultan7890/osints',
    'saved_at':      time.strftime('%Y-%m-%d %H:%M:%S'),
    'labels':        THREAT_LABELS,
    'threat_types':  THREAT_TYPES,
    'action_hints':  ACTION_HINTS,
    'best_f1':       best_f1,
    'trained_on':    str(DEVICE),
    'train_samples': len(tr_s),
    'total_samples': len(samples),
}
torch.save(checkpoint, 'sentinel_threat_net.pt')
tokenizer.save('sentinel_vocab.json')

print(f'Version      : 4.0')
print(f'Architecture : CNN + Transformer (no BiLSTM)')
print(f'Best F1      : {best_f1:.4f}')
print(f'Params       : {model.info()["params"]:,}')
print(f'Size         : {model.info()["size_mb"]} MB')
print(f'Samples      : {len(samples)}')
print(f'Outputs      : label + threat_type + action_hint + confidence + reasoning')
print()
print("Download: /kaggle/working/sentinel_threat_net.pt")
print("Download: /kaggle/working/sentinel_vocab.json")
print('Downloaded!')
print()
print('Local machine pe run karo:')
print('cp ~/Downloads/sentinel_threat_net.pt /home/kali/osints/models/ml_engine/')
print('cp ~/Downloads/sentinel_vocab.json    /home/kali/osints/models/ml_engine/')

In [ ]:
# Cell 6 — Quick Inference Test (optional)
import torch.nn.functional as F

_REASONING = {
    ('CRITICAL','web_vuln'):  'Critical web vulnerability — immediate patching required',
    ('CRITICAL','breach'):    'Active credential breach — rotate all secrets immediately',
    ('CRITICAL','apt'):       'APT indicators — escalate to SOC, preserve forensics',
    ('HIGH','web_vuln'):      'High-severity web vulnerability — schedule urgent patch',
    ('HIGH','breach'):        'Credential exposure — force password reset',
    ('HIGH','recon'):         'Active recon — review firewall rules and exposed assets',
    ('MEDIUM','recon'):       'Reconnaissance activity — monitor and harden attack surface',
    ('LOW','unknown'):        'Minimal threat indicators — continue standard monitoring',
}

def predict(text):
    model.eval()
    ids = tokenizer.encode(text, MAX_LEN)
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        l_l, l_t, l_a = model(x)
        p_l = F.softmax(l_l, dim=-1)[0]
        p_t = F.softmax(l_t, dim=-1)[0]
        p_a = F.softmax(l_a, dim=-1)[0]
    label       = IDX2LABEL[p_l.argmax().item()]
    threat_type = IDX2TYPE[p_t.argmax().item()]
    action_hint = IDX2HINT[p_a.argmax().item()]
    confidence  = round(p_l.max().item(), 4)
    reasoning   = _REASONING.get((label, threat_type),
                    f'{label} {threat_type} — confidence {confidence:.0%}')
    return {'label': label, 'threat_type': threat_type,
            'action_hint': action_hint, 'confidence': confidence,
            'reasoning': reasoning}

tests = [
    'ransomware encrypted files bitcoin ransom demand critical infrastructure',
    'sql injection vulnerability found in login form user database exposed',
    'subdomain enumeration recon 50 subdomains found cloud assets exposed',
    'normal website traffic no suspicious activity detected',
]
for t in tests:
    r = predict(t)
    print(f'[{r["label"]:8s}] type={r["threat_type"]:12s} action={r["action_hint"]:18s} conf={r["confidence"]:.2f}')
    print(f'         {r["reasoning"]}')
    print()